# 🇮🇹 Everyday Italian — Pimsleur-style audio lesson generator (B1)

**Lesson 4:** police stop (speeding, red light) · airport & airline staff (excess baggage, delays, lost luggage) · car rental (insurance, *franchigia*, ZTL) · hotel check-in/out · summer house & long-term flat rental · farmers' market · butcher · supermarket

Grammar in context: imperfect for ongoing past (*andava a novanta*), *pensavo che* + imperfect subjunctive (*fosse*), *se* + present, *è necessario che* + subjunctive, *da* + time, combined pronouns (*me lo, gliela*), conditional requests. Part 7 reviews sentences from Lessons 1–3.

How the audio works:
* **English cue → pause → Italian answer → pause.** The first pause is for you to *say* the sentence, the second for you to *repeat* after the native speaker.
* **Pause lengths depend on the Italian.** Each pause is computed from the measured duration of the Italian clip:
  * answer pause = `PAUSE_SCALE × clamp(THINK_MS + SAY_FACTOR × d_it)`
  * repeat pause = `PAUSE_SCALE × clamp(REPEAT_BASE_MS + REPEAT_FACTOR × d_it)`
* **Two Italian voices** (female/male, Microsoft neural voices via `edge-tts`) plus an English narrator; falls back to `gTTS` automatically if a clip fails.
* Back-chaining (`ingegneria → gegneria → ...`), repeated words (`x2`, `x3`, optional slow repeat), spaced review of earlier sentences, role-play, and the dialogue at start and end.

**Run:** Runtime → Run all. The MP3 and a time-stamped transcript are downloaded at the end (~3–5 min synthesis for ~420 clips; the cache makes reruns fast).


In [ ]:
!pip -q install edge-tts gTTS pydub
# ffmpeg is pre-installed on Colab (pydub needs it)

## 1 · Settings

In [ ]:
#@title Voices, speed and pause settings { display-mode: "form" }
LESSON_NAME   = "Lesson04_Viaggio_B1"  #@param {type:"string"}
ENGINE        = "edge"   #@param ["edge", "gtts"]
VOICE_EN      = "en-US-AndrewNeural"  #@param ["en-US-AndrewNeural", "en-US-GuyNeural", "en-US-JennyNeural", "en-GB-RyanNeural"]
VOICE_IT_F    = "it-IT-ElsaNeural"    #@param ["it-IT-ElsaNeural", "it-IT-IsabellaNeural"]
VOICE_IT_M    = "it-IT-DiegoNeural"   #@param ["it-IT-DiegoNeural", "it-IT-GiuseppeNeural"]
EN_RATE       = "-5%"    #@param {type:"string"}
IT_RATE       = "-10%"   #@param {type:"string"}
SLOW_RATE     = "-35%"   #@param {type:"string"}

#@markdown **Pauses** (ms).  `d_it` = duration of the Italian clip.
PAUSE_SCALE    = 1.0    #@param {type:"slider", min:0.6, max:1.8, step:0.05}
THINK_MS       = 1500   #@param {type:"integer"}
SAY_FACTOR     = 1.5    #@param {type:"number"}
MIN_ANSWER_MS  = 2500   #@param {type:"integer"}
MAX_ANSWER_MS  = 12000  #@param {type:"integer"}
REPEAT_BASE_MS = 1000   #@param {type:"integer"}
REPEAT_FACTOR  = 1.3    #@param {type:"number"}
MIN_REPEAT_MS  = 1800   #@param {type:"integer"}
MAX_REPEAT_MS  = 10000  #@param {type:"integer"}
USE_PIANO      = True   #@param {type:"boolean"}

CFG = dict(ENGINE=ENGINE, VOICE_EN=VOICE_EN, VOICE_IT_F=VOICE_IT_F, VOICE_IT_M=VOICE_IT_M,
           EN_RATE=EN_RATE, IT_RATE=IT_RATE, SLOW_RATE=SLOW_RATE, PAUSE_SCALE=PAUSE_SCALE,
           THINK_MS=THINK_MS, SAY_FACTOR=SAY_FACTOR, MIN_ANSWER_MS=MIN_ANSWER_MS, MAX_ANSWER_MS=MAX_ANSWER_MS,
           REPEAT_BASE_MS=REPEAT_BASE_MS, REPEAT_FACTOR=REPEAT_FACTOR,
           MIN_REPEAT_MS=MIN_REPEAT_MS, MAX_REPEAT_MS=MAX_REPEAT_MS,
           NARRATION_GAP_MS=700, DIALOGUE_GAP_MS=600, SECTION_GAP_MS=1200,
           # dry-run speech-rate model (calibrated on the edge-tts voices above)
           IT_SYL_PER_S=4.6, EN_WORDS_PER_S=2.7, CLIP_PAD_MS=570,
           CACHE_DIR="/content/tts_cache", USE_PIANO=USE_PIANO,
           PIANO_URL="https://csee.umbc.edu/~simsek/shortpiano.mp3")

## 2 · TTS engine

In [ ]:
import os, re, json, time, hashlib, subprocess, tempfile, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pydub import AudioSegment

# ----------------------------------------------------------------------------
# Script tags (one per line; blank lines and lines starting with # are ignored)
#   (section) Title                 -> marker in the transcript only (no audio)
#   (narrator) English text         -> English narrator. {curly braces} = Italian
#                                      inserted inside the sentence, e.g.
#                                      "The word {ricevimento} means office hours."
#   (it F) testo  / (it M) testo    -> Italian line (dialogue), female / male voice
#   (repeat [x2] [slow] [F|M]) testo-> Italian model + pause for you to REPEAT.
#                                      x2/x3 = say it 2-3 times; slow = the extra
#                                      repetitions are spoken slowly.
#   (prompt [x2] [F|M]) English || Italiano
#                                   -> English cue, pause to ANSWER (length set by
#                                      the Italian), Italian answer, pause to REPEAT.
#   (build) zione | iscrizione      -> back-chaining: each piece + repeat pause
#   (pause 3)                       -> silence in seconds
#   (english) / (italian)           -> old tags still work (= narrator / it F)
# ----------------------------------------------------------------------------

VOWELS = "aeiouàèéìíòóùúy"

def italian_syllables(text):
    """Rough Italian syllable count (vowel groups); used for dry-run timing only."""
    words = re.findall(r"[A-Za-zÀ-ÿ']+", text.lower())
    n = 0
    for w in words:
        n += max(1, len(re.findall(f"[{VOWELS}]+", w)))
    return n


class PimsleurTTS:
    def __init__(self, cfg):
        self.cfg = cfg
        os.makedirs(cfg["CACHE_DIR"], exist_ok=True)
        self.piano_clip_path = self._download_piano_clip() if cfg.get("USE_PIANO") else None

    # ------------------------------------------------------------------ parsing
    def parse(self, script):
        events = []          # list of (kind, payload dict)
        voice_toggle = 0
        for raw in script.split("\n"):
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            m = re.match(r"^\(([^)]*)\)\s*(.*)$", line)
            if not m:
                print(f"⚠️ untagged line skipped: {line[:60]}")
                continue
            tag, body = m.group(1).strip().lower(), m.group(2).strip()
            words = tag.split()
            head = words[0]
            reps = 1
            for w in words:
                if re.fullmatch(r"x\d", w):
                    reps = int(w[1])
            slow = "slow" in words
            gender = "M" if "m" in words else ("F" if "f" in words else None)

            if head in ("section",):
                events.append(("section", {"title": body}))
            elif head in ("narrator", "english", "en"):
                events.append(("narrator", {"text": body}))
            elif head in ("it", "italian", "italiano"):
                events.append(("it", {"text": body, "g": gender or "F"}))
            elif head == "repeat":
                if gender is None:
                    gender = "FM"[voice_toggle % 2]; voice_toggle += 1
                events.append(("repeat", {"text": body, "g": gender, "reps": reps, "slow": slow}))
            elif head == "prompt":
                if "||" not in body:
                    print(f"⚠️ prompt without '||' skipped: {body[:60]}"); continue
                en, it = [s.strip() for s in body.split("||", 1)]
                if gender is None:
                    gender = "FM"[voice_toggle % 2]; voice_toggle += 1
                events.append(("prompt", {"en": en, "it": it, "g": gender, "reps": reps, "slow": slow}))
            elif head == "build":
                pieces = [p.strip().lstrip("-") for p in body.split("|") if p.strip()]
                events.append(("build", {"pieces": pieces, "g": gender or "F"}))
            elif head == "pause":
                sec = float(words[1]) if len(words) > 1 else float(body or 1)
                events.append(("pause", {"ms": int(sec * 1000)}))
            else:
                print(f"⚠️ unknown tag ({tag}) skipped")
        return events

    # ------------------------------------------------------------------ pauses
    def answer_pause(self, it_ms):
        c = self.cfg
        p = c["THINK_MS"] + c["SAY_FACTOR"] * it_ms
        return int(c["PAUSE_SCALE"] * min(max(p, c["MIN_ANSWER_MS"]), c["MAX_ANSWER_MS"]))

    def repeat_pause(self, it_ms):
        c = self.cfg
        p = c["REPEAT_BASE_MS"] + c["REPEAT_FACTOR"] * it_ms
        return int(c["PAUSE_SCALE"] * min(max(p, c["MIN_REPEAT_MS"]), c["MAX_REPEAT_MS"]))

    # ------------------------------------------------------------------ TTS jobs
    def _voice(self, lang, g):
        c = self.cfg
        if lang == "en":
            return c["VOICE_EN"]
        return c["VOICE_IT_M"] if g == "M" else c["VOICE_IT_F"]

    @staticmethod
    def _split_narration(text):
        """'The word {casa} means house' -> [('en','The word'),('it','casa'),('en','means house')]"""
        parts = []
        for i, chunk in enumerate(re.split(r"\{(.*?)\}", text)):
            chunk = chunk.strip()
            if re.search(r"\w", chunk):          # skip punctuation-only pieces
                parts.append(("it" if i % 2 else "en", chunk))
        return parts

    def jobs_for(self, events):
        """All unique (text, lang, gender, slow) clips the lesson needs."""
        jobs = set()
        for kind, d in events:
            if kind == "narrator":
                for lang, chunk in self._split_narration(d["text"]):
                    jobs.add((chunk, lang, "F", False))
            elif kind == "it":
                jobs.add((d["text"], "it", d["g"], False))
            elif kind in ("repeat", "prompt"):
                it = d["text"] if kind == "repeat" else d["it"]
                jobs.add((it, "it", d["g"], False))
                if d["slow"]:
                    jobs.add((it, "it", d["g"], True))
                if kind == "prompt":
                    jobs.add((d["en"], "en", "F", False))
            elif kind == "build":
                for p in d["pieces"]:
                    jobs.add((p, "it", d["g"], False))
        return sorted(jobs)

    def _cache_path(self, job):
        text, lang, g, slow = job
        key = json.dumps([text, self._voice(lang, g), slow, self.cfg["ENGINE"],
                          self.cfg["EN_RATE"] if lang == "en" else self.cfg["IT_RATE"],
                          self.cfg["SLOW_RATE"]])
        return os.path.join(self.cfg["CACHE_DIR"], hashlib.md5(key.encode()).hexdigest() + ".mp3")

    def _synth_edge(self, job, path):
        text, lang, g, slow = job
        rate = self.cfg["SLOW_RATE"] if slow else (self.cfg["EN_RATE"] if lang == "en" else self.cfg["IT_RATE"])
        cmd = ["edge-tts", f"--voice={self._voice(lang, g)}", f"--rate={rate}",
               f"--text={text}", f"--write-media={path}"]
        proxy = os.environ.get("HTTPS_PROXY") or os.environ.get("https_proxy")
        if proxy:                       # only matters behind a proxy; Colab has none
            cmd.insert(1, f"--proxy={proxy}")
        subprocess.run(cmd, check=True, capture_output=True, timeout=60)

    def _synth_gtts(self, job, path):
        from gtts import gTTS
        text, lang, g, slow = job
        gTTS(text=text, lang=lang, slow=slow).save(path)

    @staticmethod
    def _clip_ok(job, path):
        """Reject empty / truncated clips (the online TTS sometimes returns silence)."""
        if not (os.path.exists(path) and os.path.getsize(path) > 0):
            return False
        try:
            a = AudioSegment.from_file(path)
        except Exception:
            return False
        n = italian_syllables(job[0]) if job[1] == "it" else 1.5 * len(job[0].split())
        return len(a) >= 1000 * n / 12 and a.dBFS > -60     # far faster than any voice = broken

    def _synth_one(self, job):
        path = self._cache_path(job)
        if self._clip_ok(job, path):
            return job, path, None
        err = None
        engines = [self.cfg["ENGINE"]] + (["gtts"] if self.cfg["ENGINE"] != "gtts" else [])
        for eng in engines:
            for attempt in range(4):
                try:
                    (self._synth_edge if eng == "edge" else self._synth_gtts)(job, path)
                    if self._clip_ok(job, path):
                        return job, path, None
                    err = f"{eng}: empty or truncated audio"
                except Exception as e:
                    err = f"{eng}: {e}"
                time.sleep(2 * (attempt + 1))      # back off: the free TTS service rate-limits bursts
        return job, None, err

    def synthesize_all(self, events, workers=6):
        jobs = self.jobs_for(events)
        print(f"🗣️  {len(jobs)} unique clips to synthesize (cached ones are reused)...")
        self.clips, self.failed = {}, []
        with ThreadPoolExecutor(workers) as ex:
            for i, (job, path, err) in enumerate(ex.map(self._synth_one, jobs), 1):
                if path:
                    self.clips[job] = AudioSegment.from_file(path)
                else:
                    self.failed.append((job, err))
                    self.clips[job] = AudioSegment.silent(duration=1000)
                if i % 100 == 0 or i == len(jobs):
                    print(f"   {i}/{len(jobs)} done")
        if self.failed:
            print(f"⚠️ {len(self.failed)} clips failed (replaced by 1 s silence), e.g.: {self.failed[0]}")

    def clip(self, text, lang, g="F", slow=False):
        return self.clips[(text, lang, g, slow)]

    # ------------------------------------------------------------------ assembly
    def assemble(self, events):
        c = self.cfg
        S = AudioSegment.silent
        out = S(duration=300)
        transcript = []

        def stamp(label, text):
            ms = len(out)
            transcript.append(f"[{ms//60000:02d}:{(ms//1000)%60:02d}] {label:9s} {text}")

        for kind, d in events:
            if kind == "section":
                stamp("SECTION", "=== " + d["title"] + " ===")
                out += S(duration=c["SECTION_GAP_MS"])
            elif kind == "pause":
                out += S(duration=int(d["ms"] * c["PAUSE_SCALE"]))
            elif kind == "narrator":
                stamp("NARRATOR", d["text"])
                for lang, chunk in self._split_narration(d["text"]):
                    out += self.clip(chunk, lang) + S(duration=120)
                out += S(duration=c["NARRATION_GAP_MS"])
            elif kind == "it":
                stamp(f"IT-{d['g']}", d["text"])
                out += self.clip(d["text"], "it", d["g"]) + S(duration=c["DIALOGUE_GAP_MS"])
            elif kind == "repeat":
                stamp("REPEAT", d["text"] + (f"  (x{d['reps']})" if d["reps"] > 1 else ""))
                normal = self.clip(d["text"], "it", d["g"])
                for r in range(d["reps"]):
                    seg = self.clip(d["text"], "it", d["g"], True) if (d["slow"] and r > 0) else normal
                    out += seg + S(duration=self.repeat_pause(len(seg)))
            elif kind == "prompt":
                stamp("PROMPT", d["en"] + "  ->  " + d["it"])
                normal = self.clip(d["it"], "it", d["g"])
                out += self.clip(d["en"], "en") + S(duration=self.answer_pause(len(normal)))
                for r in range(d["reps"]):
                    seg = self.clip(d["it"], "it", d["g"], True) if (d["slow"] and r > 0) else normal
                    out += seg + S(duration=self.repeat_pause(len(seg)))
            elif kind == "build":
                stamp("BUILD", " | ".join(d["pieces"]))
                for p in d["pieces"]:
                    seg = self.clip(p, "it", d["g"])
                    out += seg + S(duration=self.repeat_pause(len(seg)))
        return out, transcript

    # ------------------------------------------------------------------ dry run
    def estimate(self, events):
        """Duration estimate without any TTS (speech-rate model)."""
        c = self.cfg
        pad = c["CLIP_PAD_MS"]          # TTS clips carry some silence at both ends
        it_ms = lambda t, slow=False: pad + 1000 * italian_syllables(t) / (c["IT_SYL_PER_S"] * (0.75 if slow else 1))
        en_ms = lambda t: pad + 1000 * len(t.split()) / c["EN_WORDS_PER_S"]
        total, counts = 0.0, {}
        for kind, d in events:
            counts[kind] = counts.get(kind, 0) + 1
            if kind == "section":
                total += c["SECTION_GAP_MS"]
            elif kind == "pause":
                total += d["ms"] * c["PAUSE_SCALE"]
            elif kind == "narrator":
                for lang, chunk in self._split_narration(d["text"]):
                    total += (en_ms(chunk) if lang == "en" else it_ms(chunk)) + 120
                total += c["NARRATION_GAP_MS"]
            elif kind == "it":
                total += it_ms(d["text"]) + c["DIALOGUE_GAP_MS"]
            elif kind in ("repeat", "prompt"):
                it = d["text"] if kind == "repeat" else d["it"]
                if kind == "prompt":
                    total += en_ms(d["en"]) + self.answer_pause(it_ms(it))
                for r in range(d["reps"]):
                    m = it_ms(it, d["slow"] and r > 0)
                    total += m + self.repeat_pause(m)
            elif kind == "build":
                for p in d["pieces"]:
                    m = it_ms(p); total += m + self.repeat_pause(m)
        return total / 1000, counts

    # ------------------------------------------------------------------ main
    def _download_piano_clip(self):
        url = self.cfg.get("PIANO_URL")
        try:
            print("🎹 Downloading piano intro/outro clip...")
            p = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False).name
            urllib.request.urlretrieve(url, p)
            return p
        except Exception as e:
            print(f"⚠️ Could not download piano clip: {e}")
            return None

    def build(self, script, output_file):
        events = self.parse(script)
        self.synthesize_all(events)
        print("🎧 Assembling lesson...")
        audio, transcript = self.assemble(events)
        if self.piano_clip_path and os.path.exists(self.piano_clip_path):
            piano = AudioSegment.from_file(self.piano_clip_path)[:10000]   # first 10 s
            intro = piano.fade_in(1500).fade_out(1500)
            outro = piano.fade_in(1000).fade_out(3000)
            audio = intro + AudioSegment.silent(duration=500) + audio + AudioSegment.silent(duration=800) + outro
        audio.export(output_file, format="mp3", bitrate="96k")
        txt = os.path.splitext(output_file)[0] + "_transcript.txt"
        with open(txt, "w", encoding="utf-8") as f:
            f.write("\n".join(transcript))
        print(f"✅ {output_file}: {len(audio)/60000:.1f} min   |   transcript: {txt}")
        return output_file, txt


## 3 · Lesson script

One line = one audio event. Tags:

| tag | what you hear |
|---|---|
| `(narrator) text {italiano}` | English narrator; text in `{ }` is read by the Italian voice |
| `(it F) …` / `(it M) …` | Italian line (dialogue), female / male |
| `(repeat [x2] [slow] [F/M]) …` | Italian model + repeat pause, 1–3 times; `slow` = extra repetitions spoken slowly |
| `(prompt [x2] [F/M]) English \|\| Italiano` | English cue → answer pause → Italian → repeat pause |
| `(build) zione \| iscrizione` | back-chaining, each piece + repeat pause |
| `(pause 2)` · `(section) Title` | silence · transcript marker |

Voices for `repeat`/`prompt` alternate F/M automatically unless `F` or `M` is given.

In [ ]:
LESSON = r"""
#####################################################################
# EVERYDAY ITALIAN — Lesson 4 (B1)
# Theme: police stop (speeding, red light), airport & airline staff,
#        car rental, hotel check-in/out, summer house & long-term rental,
#        farmers' market, butcher, supermarket
# Grammar: imperfect for ongoing past (andava a novanta), pensavo che + congiuntivo
#          imperfetto, se + present -> present/future, è necessario che + congiuntivo,
#          da + time, combined pronouns (me lo, glielo), conditional requests
#####################################################################

(section) Opening
(narrator) Everyday Italian. Lesson four.
(narrator) Today you are on a trip in Italy. You will talk to airline staff, rent a car, check in to a hotel, and rent a house. You will shop at the market, at the butcher's, and at the supermarket.
(narrator) But first, a problem on the road. Marco is driving a rental car. A police officer stops him.
(narrator) As always, answer out loud in the pause. Then repeat after the Italian speaker.
(pause 1)
(it F) Buongiorno. Patente e libretto, prego.
(it M) Buongiorno. Ecco la patente. La macchina è a noleggio. Ecco il contratto.
(it F) Lo sa che andava a novanta all'ora? Qui il limite è cinquanta.
(it M) Mi scusi, non ho visto il cartello. Non conosco bene la strada.
(it F) E ha anche passato il semaforo rosso, all'incrocio.
(it M) Davvero? Pensavo che fosse ancora giallo.
(it F) No, era rosso. Purtroppo devo farle una multa.
(it M) Capisco. Quanto tempo ho per pagare?
(it F) Sessanta giorni. Ma se paga entro cinque giorni, c'è uno sconto del trenta per cento.
(it M) Posso pagare online?
(it F) Sì. Trova tutte le informazioni sul verbale.
(it M) Va bene. Mi dispiace molto. Non succederà più.
(it F) Guidi con prudenza. Buona giornata.
(pause 1.5)

(section) Part 1 — Stopped by the police
(narrator) Let's look at this conversation, piece by piece.
(narrator) The officer asks for two documents. The licence, {la patente}, and the car's registration document, {il libretto}.
(repeat x2 slow) Patente e libretto, prego.
(prompt) You are the officer. Say: licence and registration, please. || Patente e libretto, prego.
(narrator) A rental car is {una macchina a noleggio}. The contract is {il contratto}.
(build) leggio | noleggio | a noleggio
(prompt) Say: the car is a rental. || La macchina è a noleggio.
(prompt) Say: here is the contract. || Ecco il contratto.
(narrator) The officer uses the imperfect to describe what was happening. You were going at ninety: {andava a novanta}.
(repeat x2 slow) Lo sa che andava a novanta all'ora?
(prompt) Ask, formally: do you know you were going ninety kilometres per hour? || Lo sa che andava a novanta all'ora?
(prompt) Say: here the limit is fifty. || Qui il limite è cinquanta.
(narrator) The speed limit is {il limite di velocità}. Speeding is {l'eccesso di velocità}.
(build) cità | velocità | eccesso di velocità
(prompt) Say: I'm sorry, I didn't see the sign. Formal. || Mi scusi, non ho visto il cartello.
(prompt) Say: I don't know the road well. || Non conosco bene la strada.
(prompt) Say: here is the contract. || Ecco il contratto.

(narrator) A traffic light is {il semaforo}. An intersection is {l'incrocio}.
(build) foro | semaforo | il semaforo rosso
(prompt) The officer says, formally: you also ran the red light. || Ha anche passato il semaforo rosso.
(prompt) Ask: really? || Davvero?
(narrator) Marco says: I thought it was still yellow. After {pensavo che}, Italian uses the imperfect subjunctive: {fosse}.
(repeat x2 slow) Pensavo che fosse ancora giallo.
(prompt) Say: I thought it was still yellow. || Pensavo che fosse ancora giallo.
(prompt) Say: I thought the limit was seventy. || Pensavo che il limite fosse settanta.
(narrator) A fine is {una multa}. The official ticket is {il verbale}.
(prompt) The officer says: unfortunately I have to give you a fine. Formal. || Purtroppo devo farle una multa.
(prompt) Ask: how much time do I have to pay? || Quanto tempo ho per pagare?
(narrator) In Italy, if you pay a fine within five days, you often get thirty percent off. Listen to the structure: {se} plus present, then present.
(prompt) Say, formally: if you pay within five days, there is a thirty percent discount. || Se paga entro cinque giorni, c'è uno sconto del trenta per cento.
(prompt) Ask: can I pay online? || Posso pagare online?
(prompt) Say: I'm very sorry. It won't happen again. || Mi dispiace molto. Non succederà più.
(prompt) The officer says, formally: drive carefully. || Guidi con prudenza.
(narrator) Italian licences have points. For serious violations, you lose points. {Perdere punti}.
(prompt) Say: I lost five points on my licence. || Ho perso cinque punti sulla patente.
(prompt) Say: I thought it was still yellow. || Pensavo che fosse ancora giallo.
(prompt) Say: I didn't see the sign. || Non ho visto il cartello.

(section) Part 2 — At the airport
(narrator) Let's go back to the start of the trip. You are at the airport. The check-in desk is {il banco del check-in}. Boarding is {l'imbarco}.
(prompt) Ask: where is the check-in desk for Rome? || Dov'è il banco del check-in per Roma?
(prompt) The agent asks, formally: how many bags are you checking? Use {imbarcare}. || Quante valigie imbarca?
(prompt) Answer: only one, and a piece of hand luggage. || Solo una, e un bagaglio a mano.
(prompt) The agent says: your suitcase weighs twenty-five kilos. || La sua valigia pesa venticinque chili.
(prompt) The agent says: the limit is twenty-three. You have to pay the excess. || Il limite è ventitré. Deve pagare il supplemento.
(prompt) Ask: can I move something into the hand luggage? || Posso spostare qualcosa nel bagaglio a mano?
(prompt) Ask: from which gate does the flight leave? Use {l'uscita}. || Da quale uscita parte il volo?
(prompt) Say: here the limit is fifty. || Qui il limite è cinquanta.

(narrator) Now problems. A delay is {un ritardo}. If the flight is full, it is {overbooking}.
(prompt) Say: the flight is two hours late. || Il volo è in ritardo di due ore.
(prompt) Say: because of the delay, I will miss the connection. || Per colpa del ritardo, perderò la coincidenza.
(prompt) Ask, formally: could you put me on the next flight? || Potrebbe mettermi sul prossimo volo?
(prompt) Ask: do I have the right to a refund? Use {avere diritto a}. || Ho diritto a un rimborso?
(narrator) A lost bag is {un bagaglio smarrito}. At the baggage office, you file a report.
(build) rito | smarrito | bagaglio smarrito
(prompt) Say: my suitcase didn't arrive. || La mia valigia non è arrivata.
(prompt) Say: I would like to report a lost bag. Use {denunciare}. || Vorrei denunciare un bagaglio smarrito.
(prompt) Describe it: it is a blue suitcase, medium size. || È una valigia blu, di media grandezza.
(prompt) Ask: when will they deliver it to me? || Quando me la consegneranno?
(prompt) Answer, formally: we will send it to your hotel tomorrow. || Gliela mandiamo in albergo domani.
(prompt) Say, formally: if you pay within five days, there is a discount. || Se paga entro cinque giorni, c'è uno sconto.
(prompt) Say: the flight is two hours late. || Il volo è in ritardo di due ore.

(section) Part 3 — Renting a car
(narrator) To rent a car is {noleggiare una macchina}. The insurance is {l'assicurazione}.
(build) giare | noleggiare
(prompt) Say: I booked a car online. || Ho prenotato una macchina online.
(prompt) Say: the reservation is under the name Rossi. || La prenotazione è a nome Rossi.
(prompt) The agent asks, formally: do you have an international driving permit? || Ha la patente internazionale?
(prompt) Ask: is the insurance included? || L'assicurazione è inclusa?
(narrator) The excess, the amount you pay in case of damage, is {la franchigia}.
(prompt) Ask: how much is the excess? || Quant'è la franchigia?
(prompt) Say: I would like full insurance, without excess. || Vorrei l'assicurazione completa, senza franchigia.
(narrator) Most Italian rental cars have a manual gearbox. An automatic car is {una macchina con il cambio automatico}.
(prompt) Ask: do you have a car with an automatic gearbox? || Avete una macchina con il cambio automatico?
(narrator) The rule for fuel is often: full to full. {Pieno-pieno}.
(prompt) Say, formally: you must return it with a full tank. || Deve restituirla con il pieno.
(prompt) Say: there is a scratch on the door. || C'è un graffio sulla portiera.
(prompt) Ask, formally: could you write it on the contract? || Potrebbe scriverlo sul contratto?
(narrator) Many old city centres are limited traffic zones. It is called {la ZTL}. If you enter without a permit, you get a fine.
(prompt) Say: be careful, don't enter the ZTL! Speak informally. || Attento, non entrare nella ZTL!
(prompt) Say: I lost five points on my licence. || Ho perso cinque punti sulla patente.
(prompt) Say: my suitcase didn't arrive. || La mia valigia non è arrivata.

(section) Part 4 — At the hotel
(narrator) Now the hotel. Arrival is {il check-in}. Departure is {il check-out}. Both words are used in Italian.
(prompt) Say: good evening. I have a reservation under the name Rossi. || Buonasera. Ho una prenotazione a nome Rossi.
(prompt) The receptionist says, formally: may I have a document, please? || Mi dà un documento, per favore?
(prompt) Ask: what time is breakfast served? || A che ora si serve la colazione?
(prompt) Answer: from seven to ten. || Dalle sette alle dieci.
(prompt) Ask: is there parking for guests? || C'è un parcheggio per i clienti?
(prompt) Ask: what is the wifi password? || Qual è la password del wifi?
(narrator) Now some problems. To say something does not work, use {non funziona}.
(prompt) Say: the air conditioning doesn't work. || L'aria condizionata non funziona.
(prompt) Say: there is no hot water in the bathroom. || Non c'è acqua calda in bagno.
(prompt) Ask, formally: could you change my room? || Potrebbe cambiarmi la camera?
(prompt) Say: the room is too noisy. || La camera è troppo rumorosa.
(prompt) Ask: is the insurance included? || L'assicurazione è inclusa?

(narrator) At the end, you check out and pay. A late check-out is {un check-out posticipato}.
(prompt) Ask: can I leave the room later? || Posso lasciare la camera più tardi?
(prompt) Say: I would like to pay the bill. Use {saldare}. || Vorrei saldare il conto.
(prompt) Say: I didn't take anything from the minibar. || Non ho preso niente dal minibar.
(narrator) Most Italian cities have a tourist tax: {la tassa di soggiorno}.
(prompt) Ask: is the tourist tax included? || La tassa di soggiorno è inclusa?
(prompt) Ask: can I leave my luggage here until this afternoon? || Posso lasciare qui i bagagli fino a oggi pomeriggio?
(prompt) Say: the air conditioning doesn't work. || L'aria condizionata non funziona.
(prompt) Ask: do you have a car with an automatic gearbox? || Avete una macchina con il cambio automatico?

(section) Part 5 — Renting a house
(narrator) Now a summer house by the sea. A holiday house is {una casa vacanze}. To rent is {affittare}.
(prompt) Say: we rented a house by the sea for August. || Abbiamo affittato una casa al mare per agosto.
(prompt) Ask: how much does it cost per week? || Quanto costa a settimana?
(prompt) Ask: are sheets and towels included? || Lenzuola e asciugamani sono inclusi?
(prompt) Ask: is the final cleaning included? || Le pulizie finali sono incluse?
(prompt) Ask: how far is the beach? || Quanto dista la spiaggia?
(prompt) Answer: it's a five-minute walk. || È a cinque minuti a piedi.
(prompt) Ask: where can I pick up the keys? || Dove posso ritirare le chiavi?
(prompt) Say: I would like to pay the bill. || Vorrei saldare il conto.

(narrator) Now a long-term rental. A flat is {un appartamento}. Furnished is {arredato}. The monthly rent is {l'affitto}.
(build) dato | arredato | appartamento arredato
(prompt) Say: I am looking for a furnished flat near the centre. || Cerco un appartamento arredato vicino al centro.
(prompt) Ask: how much is the rent per month? || Quant'è l'affitto al mese?
(narrator) In a building, residents share costs. These are {le spese condominiali}. The deposit is {la caparra}, or {il deposito cauzionale}.
(prompt) Ask: are the building fees included? || Le spese condominiali sono incluse?
(prompt) Ask: how many months of deposit do you want? || Quante mensilità di deposito volete?
(prompt) Say: the landlord wants three months of deposit. Use {il proprietario}. || Il proprietario vuole tre mesi di caparra.
(narrator) Now the subjunctive after {è necessario che}.
(prompt) Say: it's necessary that you sign the contract. Formal. || È necessario che Lei firmi il contratto.
(prompt) Say: it's necessary that we register the contract. || È necessario che registriamo il contratto.
(narrator) To say how long something has been going on, use {da} and the present.
(prompt) Say: I have been living in this flat for two years. || Abito in questo appartamento da due anni.
(prompt) Ask: how much does it cost per week? || Quanto costa a settimana?
(prompt) Ask: can I leave the room later? || Posso lasciare la camera più tardi?

(section) Part 6 — Market, butcher and supermarket
(narrator) Now let's shop for food. At the farmers' market, {il mercato contadino}, farmers sell their own products. In season is {di stagione}.
(prompt) Ask: what is in season now? || Che cosa c'è di stagione adesso?
(prompt) Answer: in summer, there are tomatoes, zucchini, and peaches. || In estate ci sono pomodori, zucchine e pesche.
(prompt) Ask: are these vegetables organic? Use {biologico}. || Queste verdure sono biologiche?
(prompt) Answer: yes, we grow them ourselves. || Sì, le coltiviamo noi.
(prompt) Ask: can I taste one? || Posso assaggiarne una?
(prompt) Say: they are very sweet. I'll take a kilo. || Sono molto dolci. Ne prendo un chilo.
(prompt) Ask: how much is the rent per month? || Quant'è l'affitto al mese?

(narrator) The butcher's shop is {la macelleria}. The butcher is {il macellaio}.
(build) leria | macelleria
(prompt) Say: I would like half a kilo of minced beef. || Vorrei mezzo chilo di macinato di manzo.
(prompt) Say: and four pork chops. || E quattro braciole di maiale.
(narrator) Thin slices of meat are {fettine}. Can you cut it for me? {Me lo può tagliare?}
(prompt) Ask, formally: can you cut it into thin slices for me? || Me lo può tagliare a fettine sottili?
(prompt) Ask: is this chicken local? || Questo pollo è locale?
(prompt) Ask, formally: how do you cook it? || Come lo cucina Lei?
(prompt) Say: it's necessary that you sign the contract. || È necessario che Lei firmi il contratto.

(narrator) Finally, the supermarket. A department, or section, is {il reparto}. Offers are {le offerte}.
(prompt) Ask: excuse me, where is the dairy section? Use {latticini}. || Scusi, dov'è il reparto latticini?
(prompt) Say: this week, pasta is on offer. || Questa settimana la pasta è in offerta.
(prompt) Say: buy one, get one free. Use {prendi} and {paghi}. || Prendi due, paghi uno.
(narrator) In Italian supermarkets, you weigh fruit yourself, and you wear plastic gloves.
(prompt) Say: you have to weigh the fruit and put on the gloves. Use the impersonal {bisogna}. || Bisogna pesare la frutta e mettere i guanti.
(prompt) The cashier asks, formally: do you have the loyalty card? || Ha la tessera fedeltà?
(prompt) Ask: can I use the self-checkout? Use {la cassa automatica}. || Posso usare la cassa automatica?
(prompt) Say: I brought my own bags. || Ho portato le mie borse.
(prompt) Ask: what is in season now? || Che cosa c'è di stagione adesso?
(prompt) Ask, formally: can you cut it into thin slices for me? || Me lo può tagliare a fettine sottili?

(section) Part 7 — Review
(narrator) Let's review. First, sentences from earlier lessons.
(prompt) Say: I would like to register for your course, but the deadline has already passed. || Vorrei iscrivermi al suo corso, ma la scadenza è già passata.
(prompt) Ask: if it doesn't fit me, can I exchange it? || Se non mi va bene, posso cambiarla?
(prompt) Say: I would like to book a table for four people. || Vorrei prenotare un tavolo per quattro persone.
(prompt) Say: I moved here last month. || Mi sono trasferito qui il mese scorso.
(prompt) Say: the electricity bill is due on the twentieth. || La bolletta della luce scade il venti.
(prompt) Say: I missed the connection. || Ho perso la coincidenza.
(narrator) Now today's lesson. Answer quickly.
(prompt) Say: I would like to report a lost bag. || Vorrei denunciare un bagaglio smarrito.
(prompt) Ask, formally: could you put me on the next flight? || Potrebbe mettermi sul prossimo volo?
(prompt) Say: I would like full insurance, without excess. || Vorrei l'assicurazione completa, senza franchigia.
(prompt) Say: there is a scratch on the door. || C'è un graffio sulla portiera.
(prompt) Say: I have a reservation under the name Rossi. || Ho una prenotazione a nome Rossi.
(prompt) Ask, formally: could you change my room? || Potrebbe cambiarmi la camera?
(prompt) Ask: are the building fees included? || Le spese condominiali sono incluse?
(prompt) Say: I have been living in this flat for two years. || Abito in questo appartamento da due anni.
(prompt) Say: I would like half a kilo of minced beef. || Vorrei mezzo chilo di macinato di manzo.
(prompt) Say: this week, pasta is on offer. || Questa settimana la pasta è in offerta.

(narrator) Now you are Marco. Answer the police officer.
(it F) Buongiorno. Patente e libretto, prego.
(prompt M) Say: good morning. Here is the licence. The car is a rental. || Buongiorno. Ecco la patente. La macchina è a noleggio.
(it F) Lo sa che andava a novanta all'ora? Qui il limite è cinquanta.
(prompt M) Say: I'm sorry, I didn't see the sign. || Mi scusi, non ho visto il cartello.
(it F) E ha anche passato il semaforo rosso.
(prompt M) Say: really? I thought it was still yellow. || Davvero? Pensavo che fosse ancora giallo.
(it F) No, era rosso. Purtroppo devo farle una multa.
(prompt M) Ask: how much time do I have to pay? Can I pay online? || Quanto tempo ho per pagare? Posso pagare online?
(it F) Sì, trova tutto sul verbale. Guidi con prudenza.
(prompt M) Say: I'm very sorry. It won't happen again. || Mi dispiace molto. Non succederà più.

(narrator) Now listen to the first conversation again.
(pause 1)
(it F) Buongiorno. Patente e libretto, prego.
(it M) Buongiorno. Ecco la patente. La macchina è a noleggio. Ecco il contratto.
(it F) Lo sa che andava a novanta all'ora? Qui il limite è cinquanta.
(it M) Mi scusi, non ho visto il cartello. Non conosco bene la strada.
(it F) E ha anche passato il semaforo rosso, all'incrocio.
(it M) Davvero? Pensavo che fosse ancora giallo.
(it F) No, era rosso. Purtroppo devo farle una multa.
(it M) Capisco. Quanto tempo ho per pagare?
(it F) Sessanta giorni. Ma se paga entro cinque giorni, c'è uno sconto del trenta per cento.
(it M) Posso pagare online?
(it F) Sì. Trova tutte le informazioni sul verbale.
(it M) Va bene. Mi dispiace molto. Non succederà più.
(it F) Guidi con prudenza. Buona giornata.
(pause 1.5)
(narrator) This is the end of lesson four. Listen to it again tomorrow. {Buon viaggio, e guidi con prudenza!}
"""

## 4 · Dry run (no audio) — check length before synthesizing

In [ ]:
tts = PimsleurTTS(CFG)
events = tts.parse(LESSON)
sec, counts = tts.estimate(events)
print(f"Estimated length: {sec/60:.1f} min   (PAUSE_SCALE={PAUSE_SCALE})")
print(counts)
print(f"Unique clips to synthesize: {len(tts.jobs_for(events))}")

## 5 · Build the MP3 + transcript

In [ ]:
mp3, txt = tts.build(LESSON, f"{LESSON_NAME}.mp3")
from IPython.display import Audio, display
display(Audio(mp3))
try:
    from google.colab import files
    files.download(mp3); files.download(txt)
except ImportError:
    pass

## 6 · Making the next lesson

Copy this notebook, change `LESSON_NAME`, and replace the `LESSON` script. A prompt that works well with an LLM:

> Write a ~45-minute Pimsleur-style Italian lesson at level **B1** on *[topic]*, in this tag format: `(narrator)`, `(it F)`, `(it M)`, `(repeat x2 slow)`, `(prompt) English || Italiano`, `(build) a | b | c`, `(pause n)`, `(section)`. Narration in plain short English sentences; put Italian words inside narration in `{braces}`. Start with a 10–14-line dialogue, break it down with back-chaining for long words, ~120 prompts, re-prompt each new sentence 2–4 times at growing intervals (spaced recall), recycle 10–15 sentences from Lesson *N–1*, include one role-play, and replay the dialogue at the end.

Tips
* Too fast/slow for you? Move `PAUSE_SCALE` (it scales every learner pause) — no need to edit the script.
* Long sentences cap at `MAX_ANSWER_MS`; raise it if you need more time on long prompts.
* `edge-tts` is an unofficial client of the Microsoft Edge read-aloud service; if it breaks, set `ENGINE="gtts"` (single Italian voice, no male/female distinction).
